In [1]:
# ============================================================
# MiniGPT للشعر العربي - نسخة محسنة
# جاهز على Google Colab
# ============================================================

import math
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tqdm.auto import tqdm

# ---------------------------
# 1) الإعدادات
# ---------------------------
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# إعدادات الداتا
NUM_POEMS = 80000          # عدد القصائد (تقدر تزود أو تقلل)

# إعدادات الموديل
batch_size = 64
block_size = 128           # أصغر شوية عشان يشتغل أسرع على Colab
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.15

# إعدادات التدريب
max_iters = 12000
learning_rate = 3e-4
min_lr = 3e-5
warmup_iters = 500
grad_clip = 1.0
eval_interval = 500
eval_iters = 100

# ---------------------------
# 2) تنظيف العربي
# ---------------------------
def clean_arabic(text):
    if not isinstance(text, str):
        return ""

    # إزالة التشكيل والتطويل
    text = re.sub(r"[\u064B-\u0652\u0670\u0640]", "", text)

    # توحيد الألف والياء
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")

    # إزالة المسافات الزائدة
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()

# ---------------------------
# 3) تحميل داتا الشعر
# ---------------------------
def load_poetry_data(num_poems=NUM_POEMS):
    print("جاري تحميل داتا الشعر من arbml/ashaar ...")

    ds = load_dataset("arbml/ashaar", split="train", streaming=True)

    poems = []
    count = 0

    for row in tqdm(ds, total=num_poems):
        if count >= num_poems:
            break

        # استخراج النص
        verses = row.get("poem verses", []) or row.get("poem_verses", [])
        title = clean_arabic(row.get("poem title", "") or row.get("poem_title", ""))

        if not verses:
            continue

        # تحويل الأبيات لنص واحد
        if isinstance(verses, list):
            poem_text = "\n".join([clean_arabic(v) for v in verses if v])
        else:
            poem_text = clean_arabic(str(verses))

        if len(poem_text) < 40:  # تجاهل القصائد القصيرة جدًا
            continue

        # شكل موحد للداتا
        if title:
            sample = f"شعر:\nالعنوان: {title}\n{poem_text}\n\n"
        else:
            sample = f"شعر:\n{poem_text}\n\n"

        poems.append(sample)
        count += 1

    text = "".join(poems)
    print(f"\nتم تحميل {len(poems):,} قصيدة")
    print(f"إجمالي الحروف: {len(text):,}")

    return text

# ---------------------------
# 4) تحميل الداتا وبناء الـ Vocabulary
# ---------------------------
text = load_poetry_data()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s if c in stoi]

def decode(ids):
    return "".join([itos[i] for i in ids])

print(f"حجم الـ Vocabulary: {vocab_size}")

# ---------------------------
# 5) تجهيز البيانات
# ---------------------------
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train: {len(train_data):,} | Val: {len(val_data):,}")

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# ---------------------------
# 6) الموديل (MiniGPT)
# ---------------------------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_head, head_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = self.blocks(tok_emb + pos_emb)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------
# 7) إنشاء الموديل
# ---------------------------
model = MiniGPT(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"عدد الـ Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# ---------------------------
# 8) دوال مساعدة
# ---------------------------
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ---------------------------
# 9) التدريب
# ---------------------------
print("\nبدء التدريب...\n")

for it in range(max_iters):
    lr = get_lr(it)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        print(f"step {it:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | lr {lr:.6f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

# ---------------------------
# 10) حفظ الموديل + الـ Vocabulary
# ---------------------------
torch.save({
    "model_state_dict": model.state_dict(),
    "stoi": stoi,
    "itos": itos,
    "vocab_size": vocab_size,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
}, "mini_gpt_poetry.pt")

print("\nتم حفظ الموديل: mini_gpt_poetry.pt")

Device: cuda
جاري تحميل داتا الشعر من arbml/ashaar ...


  0%|          | 0/80000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# استخدام الموديل + واجهة Gradio جميلة
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# نفس تعريف الموديل بالظبط
# ---------------------------
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.15):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = self.blocks(tok_emb + pos_emb)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------
# تحميل الموديل
# ---------------------------
print("جاري تحميل الموديل...")

checkpoint = torch.load("mini_gpt_poetry.pt", map_location=device)

vocab_size = checkpoint["vocab_size"]
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]
block_size = checkpoint["block_size"]
n_embd = checkpoint["n_embd"]
n_head = checkpoint["n_head"]
n_layer = checkpoint["n_layer"]

model = MiniGPT(vocab_size, n_embd, n_head, n_layer, block_size).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("✅ تم تحميل الموديل بنجاح")

def encode(text):
    return [stoi[c] for c in text if c in stoi]

def decode(ids):
    return "".join([itos[i] for i in ids])

# ---------------------------
# دالة التوليد
# ---------------------------
def generate_poem(prompt, max_new_tokens=300, temperature=0.75, top_k=40):
    if not prompt.strip():
        prompt = "شعر:\n"

    # نتأكد إن البرومبت يبدأ بشكل صحيح
    if not prompt.startswith("شعر"):
        prompt = "شعر:\n" + prompt

    ids = encode(prompt)
    if not ids:
        ids = [0]

    context = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        generated = model.generate(
            context,
            max_new_tokens=int(max_new_tokens),
            temperature=temperature,
            top_k=int(top_k)
        )

    full_text = decode(generated[0].tolist())
    return full_text

# ---------------------------
# واجهة Gradio
# ---------------------------
custom_css = """
.gradio-container {
    font-family: 'Segoe UI', Tahoma, Arial, sans-serif;
}
textarea {
    font-size: 18px !important;
    line-height: 1.8 !important;
    direction: rtl;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="blue")) as demo:
    gr.Markdown("""
    # 🎭 مولّد الشعر العربي
    ### MiniGPT مدرّب من الصفر على الشعر العربي
    اكتب بداية قصيدة أو عنوان، والموديل هيكمل لك.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            prompt = gr.Textbox(
                label="اكتب البداية هنا",
                placeholder="مثال:\nشعر:\nالعنوان: قف على الأطلال\nقف على الأطلال يا صاحبي",
                lines=5,
                rtl=True
            )

            with gr.Row():
                max_tokens = gr.Slider(100, 600, value=300, step=20, label="طول القصيدة")
                temperature = gr.Slider(0.5, 1.2, value=0.75, step=0.05, label="درجة الإبداع (Temperature)")
                top_k = gr.Slider(10, 100, value=40, step=5, label="Top-k")

            generate_btn = gr.Button("توليد القصيدة ✨", variant="primary", size="lg")

        with gr.Column(scale=3):
            output = gr.Textbox(
                label="القصيدة المُولَّدة",
                lines=18,
                rtl=True,
                show_copy_button=True
            )

    gr.Examples(
        examples=[
            ["شعر:\nالعنوان: قف على الأطلال"],
            ["شعر:\nالعنوان: يا ليل الصب"],
            ["شعر:\nأحببتها وهواها"],
            ["شعر:\nألا يا صبا نجد"],
        ],
        inputs=prompt
    )

    generate_btn.click(
        fn=generate_poem,
        inputs=[prompt, max_tokens, temperature, top_k],
        outputs=output
    )

demo.launch(share=True)

In [4]:
# ============================================================
# استخدام الموديل + واجهة Gradio (نسخة مصلحة)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------
# تعريف الموديل
# ---------------------------
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.15):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = self.blocks(tok_emb + pos_emb)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------------------------
# تحميل الموديل
# ---------------------------
print("جاري تحميل الموديل...")

checkpoint = torch.load("mini_gpt_poetry.pt", map_location=device)

vocab_size = checkpoint["vocab_size"]
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]
block_size = checkpoint["block_size"]
n_embd = checkpoint["n_embd"]
n_head = checkpoint["n_head"]
n_layer = checkpoint["n_layer"]

model = MiniGPT(vocab_size, n_embd, n_head, n_layer, block_size).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("✅ تم تحميل الموديل بنجاح")

def encode(text):
    return [stoi[c] for c in text if c in stoi]

def decode(ids):
    return "".join([itos[i] for i in ids])

# ---------------------------
# دالة التوليد
# ---------------------------
def generate_poem(prompt, max_new_tokens=300, temperature=0.75, top_k=40):
    if not prompt.strip():
        prompt = "شعر:\n"

    if not prompt.startswith("شعر"):
        prompt = "شعر:\n" + prompt

    ids = encode(prompt)
    if not ids:
        ids = [0]

    context = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        generated = model.generate(
            context,
            max_new_tokens=int(max_new_tokens),
            temperature=float(temperature),
            top_k=int(top_k)
        )

    return decode(generated[0].tolist())

# ---------------------------
# واجهة Gradio (مصلحة)
# ---------------------------
with gr.Blocks(title="مولد الشعر العربي") as demo:
    gr.Markdown("""
    # 🎭 مولّد الشعر العربي
    ### MiniGPT مدرّب من الصفر على الشعر العربي
    اكتب بداية قصيدة أو عنوان، والموديل هيكمل لك.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            prompt = gr.Textbox(
                label="اكتب البداية هنا",
                placeholder="مثال:\nشعر:\nالعنوان: قف على الأطلال\nقف على الأطلال يا صاحبي",
                lines=5
            )

            with gr.Row():
                max_tokens = gr.Slider(100, 600, value=300, step=20, label="طول القصيدة")
                temperature = gr.Slider(0.5, 1.2, value=0.75, step=0.05, label="درجة الإبداع")
                top_k = gr.Slider(10, 100, value=40, step=5, label="Top-k")

            generate_btn = gr.Button("توليد القصيدة ✨", variant="primary")

        with gr.Column(scale=3):
            output = gr.Textbox(
                label="القصيدة المُولَّدة",
                lines=18
            )

    gr.Examples(
        examples=[
            ["شعر:\nالعنوان: قف على الأطلال"],
            ["شعر:\nالعنوان: يا ليل الصب"],
            ["شعر:\nأحببتها وهواها"],
            ["شعر:\nألا يا صبا نجد"],
        ],
        inputs=prompt
    )

    generate_btn.click(
        fn=generate_poem,
        inputs=[prompt, max_tokens, temperature, top_k],
        outputs=output
    )

demo.launch(share=True)

جاري تحميل الموديل...
✅ تم تحميل الموديل بنجاح
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a52b4b49bb7f35960.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
